In [ ]:
%load_ext autoreload

In [ ]:
%autoreload 1

In [ ]:
import os, sys, glob, random
import pickle
import math
import multiprocessing
import itertools
import warnings
import json
import time
import numpy as np
from typing import Dict, List
from rich import print
from rich.pretty import pprint
from rich.console import Console
from rich.table import Table
from rich.progress import Progress, track
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import matplotlib.ticker as ticker
from cycler import cycler
import hist
from hist import Hist
import mplhep as hep
from tabulate import tabulate
import scipy
from scipy.special import binom
import uproot
import ROOT
from iminuit import Minuit
from iminuit.cost import LeastSquares
from jacobi import propagate

parent_dir = os.path.dirname(os.path.abspath(""))
sys.path.append(parent_dir)

%aimport plot_utils
%aimport plot_utils_extras

In [ ]:
warnings.filterwarnings("ignore")
mpl.rcParams.update({"figure.max_open_warning": 0})
mpl.rcParams["figure.facecolor"] = "white"
hep.style.use("CMS")
np.set_printoptions(suppress=True)

In [ ]:
plt.plot()
hep.style.use(hep.style.CMS)
mpl.rcParams["figure.facecolor"] = "white"

In [ ]:
print("The following output directories are available:")
!ls ../../processor_output_files | grep output_histograms

## Load plots

In [ ]:
plots_CR = plot_utils.loader(
    tag="full_analysis_Dec2024_CR"
)  # , custom_lumi=559.322, load_data=True)
plots_VR = plot_utils.loader(
    tag="full_analysis_Dec2024_VR"
)  # , custom_lumi=559.322, load_data=True)
plots_SR = plot_utils.loader(
    tag="full_analysis_Dec2024_SRs"
)  # , custom_lumi=559.322, load_data=True)
plots = {}
for dataset in plots_CR:
    # Note: need to fix this to be mergable even when data for SR is missing! (blinded...)
    # This merges two dicts!
    plots[dataset] = plots_CR[dataset] | plots_VR[dataset] | plots_SR[dataset]

In [ ]:
for mS in [125, 200, 300, 400, 500, 600, 800, 1000]:
    sample = f"GluGluToSUEP_mS{mS}.000_mPhi8.000_T32.000_modeleptonic_13TeV_2018"
    print(sample, round(plots[sample]["SR_high_temp_tight"][7j::sum].value / 5000, 5))

In [ ]:
for mS in [125, 200, 300, 400, 500, 600, 800, 1000]:
    sample = f"GluGluToSUEP_mS{mS}.000_mPhi8.000_T32.000_modehadronic_13TeV_2018"
    print(sample, round(plots[sample]["SR_high_temp_tight"][7j::sum].value / 200, 5))

In [ ]:
print(plots.keys())

In [ ]:
slice_hists = {
    "VR_loose": slice(3j, None),
    "VR_tight": slice(3j, None),
    "SR_low_temp_loose": slice(4j, None),
    "SR_low_temp_tight": slice(3j, None),
    "SR_high_temp_loose": slice(4j, None),
    "SR_high_temp_tight": slice(3j, None),
}

qcd_extrapolation = plot_utils.Extrapolation(plots["QCD_Pt_MuEnrichedPt5_2018"])
# Use the following line to extrapolate for one systematic variation (or nominal)
# qcd_extrapolation.extrapolate(slice_hists=slice_hists, syst="MuonSFDown", verbose=True)
# Use the following line to extrapolate for all systematic variations
qcd_extrapolation.fit_syst_variations(slice_hists=slice_hists, verbose=False)

In [ ]:
qcd_extrapolation.plot_fit("VR", syst="")

In [ ]:
qcd_extrapolation.plot_overlay(syst="")
qcd_extrapolation.plot_fit("SR_low_temp", syst="")
qcd_extrapolation.plot_fit("SR_high_temp", syst="")

In [ ]:
qcd_extrapolation.plot_overlay(syst="MuonSFDown")
qcd_extrapolation.plot_fit("SR_low_temp", syst="MuonSFDown")
qcd_extrapolation.plot_fit("SR_high_temp", syst="MuonSFDown")

In [ ]:
slice_hists = {
    "SR_low_temp_loose": slice(4j, None),
    "SR_low_temp_tight": slice(3j, None),
    "SR_high_temp_loose": slice(4j, None),
    "SR_high_temp_tight": slice(3j, None),
}
dy_extrapolation = plot_utils.Extrapolation(plots["DY_2018"])
dy_extrapolation.extrapolate(slice_hists=slice_hists, verbose=True)
dy_extrapolation.plot_overlay()
dy_extrapolation.plot_fit("SR_low_temp")
dy_extrapolation.plot_fit("SR_high_temp")

In [ ]:
# sample = "QCD_Pt_MuEnrichedPt5_2018"
# sample = "DY_2018"
# sample = "GluGluToSUEP_mS125.000_mPhi8.000_T32.000_modeleptonic_13TeV_2018"
sample = "GluGluToSUEP_mS125.000_mPhi1.000_T0.250_modeleptonic_13TeV_2018"
region = "CR_cb"
# region = "SR_low_temp_tight"
# region = "SR_high_temp_tight"

sample_tag = (
    sample.replace("_2018", "")
    .replace("_13TeV", "")
    .replace("mode", "")
    .replace(".000", "")
    .replace("0_", "_")
    + " - "
)
if len(sample_tag) > 10:
    sample_tag += "\n"

systematics = set()
for key in plots[sample]:
    if region in key:
        systematics.add(
            key.replace(region, "")
            .replace("extrapolation", "")
            .replace("_", "")
            .replace("Up", "")
            .replace("Down", "")
        )

for syst in sorted(systematics):
    if not syst:
        continue

    fig = plt.figure(figsize=(13, 12))
    gs = gridspec.GridSpec(4, 1, left=0.08, right=0.92, bottom=0.15)

    ax1 = plt.subplot(gs[0:3, 0])  # Top 3 rows
    ax2 = plt.subplot(gs[3, 0], sharex=ax1)  # 4th row

    hep.histplot(
        plots[sample][region],
        yerr=np.sqrt(plots[sample][region].variances()),
        label="nominal",
        color="C0",
        ax=ax1,
    )
    hep.histplot(
        plots[sample][f"{region}_{syst}Up"],
        yerr=np.sqrt(plots[sample][f"{region}_{syst}Up"].variances()),
        label="up",
        color="C1",
        ax=ax1,
    )
    hep.histplot(
        plots[sample][f"{region}_{syst}Down"],
        yerr=np.sqrt(plots[sample][f"{region}_{syst}Down"].variances()),
        label="down",
        color="C2",
        ax=ax1,
    )

    ax2.axhline(1, ls="--", color="gray")
    hep.histplot(
        np.divide(
            plots[sample][f"{region}_{syst}Up"].values(),
            plots[sample][region].values(),
            out=np.ones_like(plots[sample][region].values()),
            where=(plots[sample][region].values() > 0),
        ),
        bins=plots[sample][region].axes[0].edges,
        label="up",
        color="C1",
        ax=ax2,
    )
    hep.histplot(
        np.divide(
            plots[sample][f"{region}_{syst}Down"].values(),
            plots[sample][region].values(),
            out=np.ones_like(plots[sample][region].values()),
            where=(plots[sample][region].values() > 0),
        ),
        bins=plots[sample][region].axes[0].edges,
        label="down",
        color="C2",
        ax=ax2,
    )

    ax1.set_yscale("log")
    ax1.set_ylabel("Events")
    ax1.set_title(f"{sample_tag}{region} - {syst}")
    ax1.legend()
    ax1.xaxis.set_minor_locator(ticker.NullLocator())
    ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax2.set_ylim(0.5, 1.5)
    ax2.set_ylabel("Ratio")
    ax2.set_xlabel("nMuon")
    ax1.set_xlabel("")
    for label in ax1.xaxis.get_ticklabels():
        label.set_visible(False)
    plt.tight_layout()
    plt.show()

These are all the samples loaded.

In [ ]:
pprint(list(plots.keys()))

In [ ]:
pprint(list(plots["DY_NJets_LO_2018"].keys()))

In [ ]:
pprint(plots)

## Plot stack of bkgs for nMuon distribution

In [ ]:
from cycler import cycler

cmap_petroff_6 = ["#5790fc", "#f89c20", "#e42536", "#964a8b", "#9c9ca1", "#7a21dd"]
cmap_petroff_10 = [
    "#3f90da",
    "#ffa90e",
    "#bd1f01",
    "#94a4a2",
    "#832db6",
    "#a96b59",
    "#e76300",
    "#b9ac70",
    "#717581",
    "#92dadd",
]

CMS = {"axes.prop_cycle": cycler("color", cmap_petroff_10)}
plt.style.use(CMS)
# np.set_printoptions(precision=3)

## Let's write a function that will create the QCD extrapolation
We need to:
1. Fit the histograms and get the parameters.
2. Produce the extrapolated histogram for QCD in SR_tight.
3. Export histograms to a root file. The QCD is going to be the extrapolated histogram.

In [ ]:
pprint(plots["QCD_Pt_MuEnrichedPt5_2018"])

## This is to explore which datasets are the best

In [ ]:
# mc_processes = [
# #     'DY_NJets_LO_2018',
# #     'DY_LHEFilterPtZ_NLO_2018',
# #     'DY_M-50_inclusive_NLO_2018',
# #     'DY_M-10to50_inclusive_NLO_2018',
# #     'DY_M-10to50_inclusive_LO_2018',
# #     'QCD_Pt_MuEnrichedPt5_2018',
# #     'TTW_NLO_2018',
# #     'TTZ_NLO_2018',
# #     'TTZToLLNuNu_M-10_NLO_2018',
# #     'TTZToLL_LO_2018',
# #     'TTZ_inclusive_LO_2018',
# #     'TTTT_NLO_2018',
# #     'ST_NLO_2018',
# #     'ST_tW_powheg_2018',
#     'WJetsToLNu_HT_LO_2018',
#     'WJetsToLNu_Pt_NLO_2018',
#     'WJetsToLNu_inclusive_NLO_2018',
# #     'VV_NLO_2018',
# #     'VVV_NLO_2018',
# #     'Higgs_2018',
# ]

mc_processes = [
    #     'DY_NJets_LO_2018',
    #     'DY_LHEFilterPtZ_NLO_2018',
    #     'DY_M-10to50_inclusive_NLO_2018',
    #     'DY_M-10to50_inclusive_LO_2018',
    #     'DY_M-50_inclusive_NLO_2018',
    #     'DYToMuMu_M-10to50_NLO_2018',
    #     'DYToMuMu_M-50_NLO_2018',
    "DY_2018",
    #     'QCD_Pt_MuEnrichedPt5_2018',
    "TT_powheg_2018",
    #     'TTW_NLO_2018',
    #     'TTZ_NLO_2018',
    "TTV_2018",
    #     'TTZToLLNuNu_M-10_NLO_2018',
    #     'TTZToLL_LO_2018',
    #     'TTZ_inclusive_LO_2018',
    #     'TTTT_NLO_2018',
    #     'ST_s-channel_NLO_2018',
    #     'ST_t-channel_powheg_2018',
    #     'ST_tW_Dilept_NLO_2018',
    "ST_NLO_2018",
    #     'ST_tW_powheg_2018',
    #     'WJetsToLNu_HT_LO_2018',
    #     'WJetsToLNu_Pt_NLO_2018',
    #     'WJetsToLNu_inclusive_NLO_2018',
    "WJets_2018",
    #     'WW_NLO_2018',
    #     'WZ_NLO_2018',
    #     'ZZ_NLO_2018',
    #     'VV_NLO_2018',
    #     'VVV_NLO_2018',
    "VV+VVV_2018",
    "Higgs_2018",
    #     'WH_HToBB_powheg_2018',
    #     'ttH_powheg_2018',
    #     'GluGluHToZZTo4L_2018',
    #     'VBF_HToZZTo4L_2018',
    #     'ZH_HToZZ_4LFilter_2018',
    #     'WH_HToZZTo4L_2018',
    #     'QCD_Pt-15To20_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-20To30_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-30To50_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-50To80_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-80To120_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-120To170_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-170To300_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-300To470_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-470To600_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-600To800_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-800To1000_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'QCD_Pt-1000_MuEnrichedPt5_TuneCP5_13TeV-pythia8_2018',
    #     'TTZToQQ_TuneCP5_13TeV_amcatnlo-pythia8_2018',
    #     'DYJetsToMuMu_M-10to50_H2ErratumFix_TuneCP5_13TeV-powhegMiNNLO-pythia8-photos_2018',
    #     'DYJetsToMuMu_M-50_massWgtFix_TuneCP5_13TeV-powhegMiNNLO-pythia8-photos_2018',
    #     'DYJetsToLL_M-50_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DY1JetsToLL_M-50_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'DY2JetsToLL_M-50_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'DY3JetsToLL_M-50_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'DY4JetsToLL_M-50_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'DYJetsToLL_LHEFilterPtZ-0_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DYJetsToLL_LHEFilterPtZ-0To50_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DYJetsToLL_LHEFilterPtZ-50To100_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DYJetsToLL_LHEFilterPtZ-100To250_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DYJetsToLL_LHEFilterPtZ-250To400_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DYJetsToLL_LHEFilterPtZ-400To650_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DYJetsToLL_LHEFilterPtZ-650ToInf_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DYJetsToLL_M-10to50_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'DYJetsToLL_M-10to50_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'WJetsToLNu_Pt-100To250_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'WZTo2Q2L_mllmin4p0_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'WJetsToLNu_Pt-600ToInf_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'ZH_HToZZ_4LFilter_M125_TuneCP5_13TeV_powheg2-minlo-HZJ_JHUGenV7011_pythia8_2018',
    #     'ZZTo2Q2L_mllmin4p0_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'TTZToLLNuNu_M-10_TuneCP5_13TeV-amcatnlo-pythia8_2018',
    #     'ttHToNonbb_M125_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'WJetsToLNu_HT-1200To2500_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_minloHJJ_JHUGenV7011_pythia8_2018',
    #     'ST_tW_Dilept_5f_DR_TuneCP5_13TeV-amcatnlo-pythia8_2018',
    #     'WWW_4F_TuneCP5_13TeV-amcatnlo-pythia8_2018',
    #     'ST_tW_antitop_5f_inclusiveDecays_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'WZTo1L1Nu2Q_4f_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'TTZToQQ_TuneCP5_13TeV-amcatnlo-pythia8_2018',
    #     'WplusH_HToBB_WToLNu_M-125_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'TTToHadronic_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'ttH_HToZZ_4LFilter_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8_2018',
    #     'WZTo3LNu_mllmin4p0_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'ZZTo4L_TuneCP5_13TeV_powheg_pythia8_2018',
    #     'ST_s-channel_4f_leptonDecays_TuneCP5_13TeV-amcatnlo-pythia8_2018',
    #     'WJetsToLNu_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'WWZ_4F_TuneCP5_13TeV-amcatnlo-pythia8_2018',
    #     'WJetsToLNu_HT-400To600_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'WJetsToLNu_HT-100To200_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'TTZToLL_TuneCP5_13TeV_amcatnlo-pythia8_2018',
    #     'TTTT_TuneCP5_13TeV-amcatnlo-pythia8_2018',
    #     'TTToSemiLeptonic_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'ttZJets_TuneCP5_13TeV_madgraphMLM_pythia8_2018',
    #     'WJetsToLNu_Pt-250To400_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'ZZZ_TuneCP5_13TeV-amcatnlo-pythia8_2018',
    #     'TTTo2L2Nu_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'ttHTobb_M125_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'GluGluToZH_HToZZTo4L_M125_TuneCP5_13TeV-jhugenv723-pythia8_2018',
    #     'WWTo2L2Nu_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'WWTo1L1Nu2Q_4f_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'WJetsToLNu_HT-800To1200_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'WJetsToLNu_Pt-400To600_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8_2018',
    #     'ZZTo2L2Nu_TuneCP5_13TeV_powheg_pythia8_2018',
    #     'TTWJetsToLNu_TuneCP5_13TeV-amcatnloFXFX-madspin-pythia8_2018',
    #     'WJetsToLNu_HT-600To800_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'WJetsToLNu_HT-70To100_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'TTWJetsToQQ_TuneCP5_13TeV-amcatnloFXFX-madspin-pythia8_2018',
    #     'ST_tW_top_5f_inclusiveDecays_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'VHToNonbb_M125_TuneCP5_13TeV-amcatnloFXFX_madspin_pythia8_2018',
    #     'WJetsToLNu_HT-2500ToInf_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'TTZToLL_5f_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'WJetsToLNu_HT-200To400_TuneCP5_13TeV-madgraphMLM-pythia8_2018',
    #     'ST_t-channel_antitop_4f_InclusiveDecays_TuneCP5_13TeV-powheg-madspin-pythia8_2018',
    #     'WminusH_HToBB_WToLNu_M-125_TuneCP5_13TeV-powheg-pythia8_2018',
    #     'ST_t-channel_top_4f_InclusiveDecays_TuneCP5_13TeV-powheg-madspin-pythia8_2018',
    #     'VBF_HToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8_2018',
    #     'WminusH_HToZZTo4L_M125_TuneCP5_13TeV_powheg2-minlo-HWJ_JHUGenV7011_pythia8_2018',
    #     'WplusH_HToZZTo4L_M125_TuneCP5_13TeV_powheg2-minlo-HWJ_JHUGenV7011_pythia8_2018'
]

final_processes = [
    "DY_LHEFilterPtZ_NLO_2018",
    "DY_M-10to50_inclusive_NLO_2018",
    "QCD_Pt_MuEnrichedPt5_2018",
    "TTW_NLO_2018",
    "TTZ_NLO_2018",
    "TTTT_NLO_2018",
    "ST_NLO_2018",
    #     'WJetsToLNu_HT_LO_2018',
    #     'WJetsToLNu_Pt_NLO_2018',
    #     'WJetsToLNu_inclusive_NLO_2018',
    "VV_NLO_2018",
    "VVV_NLO_2018",
    "Higgs_2018",
]

regions = [
    "CR_prompt",
    "CR_light",
    "CR_cb",
    "VR",
    "SR_high_temp_tight",
    "SR_high_temp_loose",
    "SR_low_temp_tight",
    "SR_low_temp_loose",
]
y_limits = [
    (1e0, 1e10),
    (1e-2, 1e13),
    (1e-2, 1e13),
    (1e-2, 1e13),
    (1e-3, 1e12),
    (1e-2, 1e13),
]


for i_r, region in enumerate(regions):
    hists_mc = []
    hist_bkg_total = None

    print(region)
    for process in mc_processes:
        h_mc = plots[process][region]
        print("\t", process)
        for val, err in zip(h_mc.values(), np.sqrt(h_mc.variances())):
            print(f"\t\t{val:.3f} ± {err:.3f}")
        hists_mc.append(h_mc)
        if hist_bkg_total is None:
            hist_bkg_total = h_mc.copy()
        else:
            hist_bkg_total += h_mc.copy()

    # fig, ax = plt.subplots(figsize=(12.5,11))
    fig, ax = plt.subplots(figsize=(18, 15))

    hep.histplot(
        hists_mc,
        yerr=[np.sqrt(h.variances()) for h in hists_mc],
        label=[label.replace("_2018", "") for label in mc_processes],
        lw=3,
        ax=ax,
    )

    hep.cms.label(llabel="Preliminary", data=True, lumi=55, ax=ax)
    plt.title(region)
    plt.gca().xaxis.set_minor_locator(plt.NullLocator())
    plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    # plt.ylim(1e-4, 1e6)
    plt.ylim(1e-3, 1e8)
    # plt.ylim(0.01*y_limits[i_r][0], 0.01*y_limits[i_r][1])
    plt.yscale("log")
    plt.legend()  # ncol=2)
    plt.xlabel("nMuon")
    plt.ylabel("events")
    plt.show()

## See where signal is

In [ ]:
mc_processes = [
    "DY_2018",
    "QCD_Pt_MuEnrichedPt5_2018",
]
decay = "leptonic"
mS = 125
T = 32
mPhi = 8
mc_processes += [
    # f'GluGluToSUEP_mS{mS:.3f}_mPhi{mPhi:.3f}_T{T:.3f}_mode{decay}_13TeV_2018' for decay in ['leptonic', 'hadronic']
    # f'GluGluToSUEP_mS{mS:.3f}_mPhi{mPhi:.3f}_T{T:.3f}_mode{decay}_13TeV_2018' for mPhi, T in [(1, 0.25), (2, 2), (2, 4)]
    # f'GluGluToSUEP_mS{mS:.3f}_mPhi{mPhi:.3f}_T{T:.3f}_mode{decay}_13TeV_2018' for mPhi, T in [(4, 1), (4, 4), (4, 8), (4, 16)]
    f"GluGluToSUEP_mS{mS:.3f}_mPhi{mPhi:.3f}_T{T:.3f}_mode{decay}_13TeV_2018"
    for mPhi, T in [(8, 2), (8, 4), (8, 8), (8, 16), (8, 32)]
    # f'GluGluToSUEP_mS{mS:.3f}_mPhi{mPhi:.3f}_T{T:.3f}_mode{decay}_13TeV_2018' for mS in [125, 200, 300, 400, 500, 600, 800, 1000]
]

regions = [
    "CR_prompt",
    "CR_light",
    "CR_cb",
    "VR",
    "SR_high_temp_tight",
    "SR_high_temp_loose",
    "SR_low_temp_tight",
    "SR_low_temp_loose",
]
y_limits = [
    (1e0, 1e10),
    (1e-2, 1e13),
    (1e-2, 1e13),
    (1e-2, 1e13),
    (1e-3, 1e12),
    (1e-2, 1e13),
]

for i_r, region in enumerate(regions):
    hists_mc = []
    hist_bkg_total = None

    # print(region)
    for process in mc_processes:
        if process not in plots:
            continue
        h_mc = plots[process][region]
        # print("\t", process)
        # for val, err in zip(h_mc.values(), np.sqrt(h_mc.variances())):
        #     print(f"\t\t{val:.3f} ± {err:.3f}")
        hists_mc.append(h_mc)
        if hist_bkg_total is None:
            hist_bkg_total = h_mc.copy()
        else:
            hist_bkg_total += h_mc.copy()

    # fig, ax = plt.subplots(figsize=(12.5,11))
    fig, ax = plt.subplots(figsize=(18, 15))

    hep.histplot(
        hists_mc,
        yerr=[np.sqrt(h.variances()) for h in hists_mc],
        label=[label.replace("_2018", "") for label in mc_processes],
        lw=3,
        ax=ax,
    )

    hep.cms.label(llabel="Preliminary", data=True, lumi=55, ax=ax)
    plt.title(region)
    plt.gca().xaxis.set_minor_locator(plt.NullLocator())
    plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    # plt.ylim(1e-4, 1e6)
    plt.ylim(1e-3, 1e8)
    # plt.ylim(0.01*y_limits[i_r][0], 0.01*y_limits[i_r][1])
    plt.yscale("log")
    plt.legend()  # ncol=2)
    plt.xlabel("nMuon")
    plt.ylabel("events")
    plt.show()

In [ ]:
mc_processes = [
    ("Higgs_2018", "Higgs"),
    # ('TTV_2018', 'TTV'),
    ("ST_NLO_2018", "ST"),
    ("WJets_2018", "WJets"),
    ("VV+VVV_2018", "VV+VVV"),
    ("TT_powheg_2018", "TT"),
    # ("DY_2018", "DY+extr."),
    ("DY_2018", "DY"),
    # ("QCD_Pt_MuEnrichedPt5_2018", "QCD+extr."),
    ("QCD_Pt_MuEnrichedPt5_2018", "QCD"),
]

signal_processes = [
    "GluGluToSUEP_mS125.000_mPhi8.000_T8.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS300.000_mPhi4.000_T16.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS600.000_mPhi8.000_T16.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS1000.000_mPhi8.000_T32.000_modeleptonic_13TeV_2018",
]

hists_mc = []
hist_bkg_total = plots[mc_processes[0][0]]["SR_high_temp_tight"].copy().reset()

for process, label in mc_processes:
    h_mc = plots[process][
        "SR_high_temp_tight_extrapolation" if "extr" in label else "SR_high_temp_tight"
    ]
    hists_mc.append(h_mc)
    hist_bkg_total += h_mc.copy()

hists_signal = []
for process in signal_processes:
    h_signal = plots[process]["SR_high_temp_tight"]
    hists_signal.append(h_signal)

fig, ax = plt.subplots(figsize=(12.5, 11))

hep.histplot(
    hists_mc,
    yerr=[np.sqrt(h.variances()) for h in hists_mc],
    stack=True,
    label=[p[1] for p in mc_processes],
    histtype="fill",
    ec="black",
    lw=2,
    ax=ax,
)

x_hatch = np.vstack(
    (hist_bkg_total.axes[0].edges[:-1], hist_bkg_total.axes[0].edges[1:])
).reshape((-1,), order="F")
y_hatch1 = np.vstack((hist_bkg_total.values(), hist_bkg_total.values())).reshape(
    (-1,), order="F"
)
y_hatch1_unc = np.vstack(
    (np.sqrt(hist_bkg_total.variances()), np.sqrt(hist_bkg_total.variances()))
).reshape((-1,), order="F")
ax.fill_between(
    x=x_hatch,
    y1=y_hatch1 - y_hatch1_unc,
    y2=y_hatch1 + y_hatch1_unc,
    label="Stat. Unc.",
    step="pre",
    facecolor="none",
    edgecolor=(0, 0, 0, 0.5),
    linewidth=0,
    hatch="///",
    zorder=2,
)

hep.histplot(
    hists_signal,
    yerr=[np.sqrt(h.variances()) for h in hists_signal],
    label=[
        s.replace("GluGluTo", "")
        .replace(".000", "")
        .replace("mode", "")
        .replace("_13TeV_2018", "")
        for s in signal_processes
    ],
    lw=3,
    ls="--",
    ax=ax,
)

plt.vlines(x=7, color="red", ymin=1e-3, ymax=1e6, lw=4)
plt.annotate(
    "", xy=(7.5, 1e5), xytext=(7, 1e5), arrowprops=dict(facecolor="red", shrink=0)
)


hep.cms.label(llabel="Preliminary", data=True, lumi=55, ax=ax)
plt.text(5.5, 2e5, "SR_high_T_tight", ha="center", weight="bold")
plt.gca().xaxis.set_minor_locator(plt.NullLocator())
plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.ylim(1e-3, 1e11)
plt.yscale("log")
plt.legend(ncol=2)
plt.xlabel(r"$n_{muon}$")
plt.ylabel("events")
plt.show()

In [ ]:
mc_processes = [
    ("Higgs_2018", "Higgs"),
    # ('TTV_2018', 'TTV'),
    ("ST_NLO_2018", "ST"),
    ("WJets_2018", "WJets"),
    ("VV+VVV_2018", "VV+VVV"),
    ("TT_powheg_2018", "TT"),
    # ("DY_2018", "DY+extr."),
    ("DY_2018", "DY"),
    # ("QCD_Pt_MuEnrichedPt5_2018", "QCD+extr."),
    ("QCD_Pt_MuEnrichedPt5_2018", "QCD"),
]

signal_processes = [
    "GluGluToSUEP_mS125.000_mPhi4.000_T1.000_modehadronic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi2.000_T2.000_modehadronic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi2.000_T4.000_modehadronic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi4.000_T4.000_modehadronic_13TeV_2018",
]

hists_mc = []
hist_bkg_total = plots[mc_processes[0][0]]["SR_low_temp_tight"][:8j].copy().reset()

for process, label in mc_processes:
    h_mc = plots[process][
        "SR_low_temp_tight_extrapolation" if "extr" in label else "SR_low_temp_tight"
    ][:8j]
    hists_mc.append(h_mc)
    hist_bkg_total += h_mc.copy()

hists_signal = []
for process in signal_processes:
    h_signal = plots[process]["SR_low_temp_tight"][:8j]
    hists_signal.append(h_signal)

fig, ax = plt.subplots(figsize=(12.5, 11))

hep.histplot(
    hists_mc,
    yerr=[np.sqrt(h.variances()) for h in hists_mc],
    stack=True,
    label=[p[1] for p in mc_processes],
    histtype="fill",
    ec="black",
    lw=2,
    ax=ax,
)

x_hatch = np.vstack(
    (hist_bkg_total.axes[0].edges[:-1], hist_bkg_total.axes[0].edges[1:])
).reshape((-1,), order="F")
y_hatch1 = np.vstack((hist_bkg_total.values(), hist_bkg_total.values())).reshape(
    (-1,), order="F"
)
y_hatch1_unc = np.vstack(
    (np.sqrt(hist_bkg_total.variances()), np.sqrt(hist_bkg_total.variances()))
).reshape((-1,), order="F")
ax.fill_between(
    x=x_hatch,
    y1=y_hatch1 - y_hatch1_unc,
    y2=y_hatch1 + y_hatch1_unc,
    label="Stat. Unc.",
    step="pre",
    facecolor="none",
    edgecolor=(0, 0, 0, 0.5),
    linewidth=0,
    hatch="///",
    zorder=2,
)

hep.histplot(
    hists_signal,
    yerr=[np.sqrt(h.variances()) for h in hists_signal],
    label=[
        s.replace("GluGluTo", "")
        .replace(".000", "")
        .replace("mode", "")
        .replace("_13TeV_2018", "")
        for s in signal_processes
    ],
    lw=3,
    ls="--",
    ax=ax,
)

plt.vlines(x=7, color="red", ymin=1e-3, ymax=1e5, lw=4)
plt.annotate(
    "", xy=(7.5, 1e4), xytext=(7, 1e4), arrowprops=dict(facecolor="red", shrink=0)
)


hep.cms.label(llabel="Preliminary", data=True, lumi=55, ax=ax)
plt.text(5.5, 2e4, "SR_low_T_tight", ha="center", weight="bold")
plt.gca().xaxis.set_minor_locator(plt.NullLocator())
plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.ylim(1e-2, 1e9)
plt.yscale("log")
plt.legend(ncol=2)
plt.xlabel(r"$n_{muon}$")
plt.ylabel("events")
plt.show()

## Validation

In [ ]:
mc_processes = [
    ("Higgs_2018", "Higgs"),
    # ('TTV_2018', 'TTV'),
    ("ST_NLO_2018", "ST"),
    ("WJets_2018", "WJets"),
    ("VV+VVV_2018", "VV+VVV"),
    ("TT_powheg_2018", "TT"),
    ("DY_2018", "DY+extr."),
    # ('DY_2018', 'DY'),
    ("QCD_Pt_MuEnrichedPt5_2018", "QCD+extr."),
    # ('QCD_Pt_MuEnrichedPt5_2018', 'QCD'),
]

signal_processes = [
    "GluGluToSUEP_mS125.000_mPhi8.000_T8.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS300.000_mPhi4.000_T16.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS600.000_mPhi8.000_T16.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS1000.000_mPhi8.000_T32.000_modeleptonic_13TeV_2018",
]

hists_mc = []
hist_bkg_total = plots[mc_processes[0][0]]["VR"].copy().reset()

for process, label in mc_processes:
    h_mc = plots[process]["VR"]
    hists_mc.append(h_mc)
    hist_bkg_total += h_mc.copy()

hists_signal = []
for process in signal_processes:
    h_signal = plots[process]["VR"]
    hists_signal.append(h_signal)

fig, ax = plt.subplots(figsize=(12.5, 11))

hep.histplot(
    hists_mc,
    yerr=[np.sqrt(h.variances()) for h in hists_mc],
    stack=True,
    label=[p[1] for p in mc_processes],
    histtype="fill",
    ec="black",
    lw=2,
    ax=ax,
)

x_hatch = np.vstack(
    (hist_bkg_total.axes[0].edges[:-1], hist_bkg_total.axes[0].edges[1:])
).reshape((-1,), order="F")
y_hatch1 = np.vstack((hist_bkg_total.values(), hist_bkg_total.values())).reshape(
    (-1,), order="F"
)
y_hatch1_unc = np.vstack(
    (np.sqrt(hist_bkg_total.variances()), np.sqrt(hist_bkg_total.variances()))
).reshape((-1,), order="F")
ax.fill_between(
    x=x_hatch,
    y1=y_hatch1 - y_hatch1_unc,
    y2=y_hatch1 + y_hatch1_unc,
    label="Stat. Unc.",
    step="pre",
    facecolor="none",
    edgecolor=(0, 0, 0, 0.5),
    linewidth=0,
    hatch="///",
    zorder=2,
)

hep.histplot(
    hists_signal,
    yerr=[np.sqrt(h.variances()) for h in hists_signal],
    label=[
        s.replace("GluGluTo", "")
        .replace(".000", "")
        .replace("mode", "")
        .replace("_13TeV_2018", "")
        for s in signal_processes
    ],
    lw=3,
    ls="--",
    ax=ax,
)

plt.vlines(x=7, color="red", ymin=1e-3, ymax=1e6, lw=4)
plt.annotate(
    "", xy=(7.5, 1e5), xytext=(7, 1e5), arrowprops=dict(facecolor="red", shrink=0)
)


hep.cms.label(llabel="Preliminary", data=True, lumi=55, ax=ax)
plt.text(5.5, 2e5, "VR", ha="center", weight="bold")
plt.gca().xaxis.set_minor_locator(plt.NullLocator())
plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.ylim(1e-3, 1e11)
plt.yscale("log")
plt.legend(ncol=2)
plt.xlabel(r"$n_{muon}$")
plt.ylabel("events")
plt.show()

In [ ]:
data = "DoubleMuon+Run2018A-UL2018_MiniAODv2-v1+MINIAOD_2018"
regions = ["CR_prompt", "CR_light", "CR_cb", "SR_high_temp_tight", "SR_high_temp_loose"]
y_limits = [
    (1e0, 1e10),
    (1e-2, 1e13),
    (1e-2, 1e13),
    (1e-3, 1e12),
    (1e-2, 1e13),
]
mc_processes = [
    "VVV_NLO_2018",
    "TTZ_inclusive_LO_2018",
    "VV_NLO_2018",
    "WJetsToLNu_2018",
    "ST_NLO_2018",
    "TT_powheg_2018",
    "DY_inclusive_NLO_2018",
    "QCD_Pt_MuEnrichedPt5_2018",
]
mc_labels = [
    "VVV",
    "TTZ",
    "VV",
    "WJetsToLNu",
    "ST",
    "TT",
    "DY",
    "QCD",
]

signal_processes = [
    "GluGluToSUEP_mS125.000_mPhi8.000_T16.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi8.000_T16.000_modehadronic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi8.000_T32.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi8.000_T32.000_modehadronic_13TeV_2018",
]
signal_labels = [
    "mS125_mPhi8_T16_leptonic",
    "mS125_mPhi8_T16_hadronic",
    "mS125_mPhi8_T32_leptonic",
    "mS125_mPhi8_T32_hadronic",
]

for i_r, region in enumerate(regions):
    hists_mc = []
    hist_bkg_total = None

    for process in mc_processes:
        h_mc = plots[process][region]
        print(process, h_mc.values())
        hists_mc.append(h_mc)
        if hist_bkg_total is None:
            hist_bkg_total = h_mc.copy()
        else:
            hist_bkg_total += h_mc.copy()

    hists_signal = []
    for process in signal_processes:
        h_signal = plots[process][region]
        hists_signal.append(h_signal)

    fig, ax = plt.subplots(figsize=(12.5, 11))

    hep.histplot(
        hists_mc,
        yerr=[np.sqrt(h.variances()) for h in hists_mc],
        stack=True,
        label=mc_labels,
        histtype="fill",
        ec="black",
        lw=2,
        ax=ax,
    )

    x_hatch = np.vstack(
        (hist_bkg_total.axes[0].edges[:-1], hist_bkg_total.axes[0].edges[1:])
    ).reshape((-1,), order="F")
    y_hatch1 = np.vstack((hist_bkg_total.values(), hist_bkg_total.values())).reshape(
        (-1,), order="F"
    )
    y_hatch1_unc = np.vstack(
        (np.sqrt(hist_bkg_total.variances()), np.sqrt(hist_bkg_total.variances()))
    ).reshape((-1,), order="F")
    ax.fill_between(
        x=x_hatch,
        y1=y_hatch1 - y_hatch1_unc,
        y2=y_hatch1 + y_hatch1_unc,
        label="Stat. Unc.",
        step="pre",
        facecolor="none",
        edgecolor=(0, 0, 0, 0.5),
        linewidth=0,
        hatch="///",
        zorder=2,
    )

    hep.histplot(
        hists_signal,
        yerr=[np.sqrt(h.variances()) for h in hists_signal],
        label=signal_labels,
        lw=3,
        ls="--",
        ax=ax,
    )

    h_data = plots[data][region]
    hep.histplot(
        h_data,
        label=["Data"],
        histtype="errorbar",
        mec="black",
        mfc="black",
        ecolor="black",
        ax=ax,
    )

    hep.cms.label(llabel="Preliminary", data=True, lumi=0.56, ax=ax)
    # hep.cms.label(llabel="Preliminary", data=True, lumi=55, ax=ax)
    plt.title(region)
    plt.gca().xaxis.set_minor_locator(plt.NullLocator())
    plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    plt.ylim(0.01 * y_limits[i_r][0], 0.01 * y_limits[i_r][1])
    plt.yscale("log")
    plt.legend(ncol=2)
    plt.xlabel("nMuon")
    plt.ylabel("events")
    plt.show()

In [ ]:
data = "DoubleMuon+Run2018A-UL2018_MiniAODv2-v1+MINIAOD_2018"
regions = ["CR_prompt", "CR_light", "CR_cb", "SR_high_temp_tight", "SR_high_temp_loose"]
y_limits = [
    (1e0, 1e10),
    (1e-2, 1e13),
    (1e-2, 1e13),
    (1e-3, 1e12),
    (1e-2, 1e13),
]
mc_processes = [
    "VVV_NLO_2018",
    "TTZ_inclusive_LO_2018",
    "VV_NLO_2018",
    "WJetsToLNu_2018",
    "ST_NLO_2018",
    "TT_powheg_2018",
    "DY_inclusive_NLO_2018",
    "QCD_Pt_MuEnrichedPt5_2018",
]
mc_labels = [
    "VVV",
    "TTZ",
    "VV",
    "WJetsToLNu",
    "ST",
    "TT",
    "DY",
    "QCD",
]

signal_processes = [
    "GluGluToSUEP_mS125.000_mPhi8.000_T16.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi8.000_T16.000_modehadronic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi8.000_T32.000_modeleptonic_13TeV_2018",
    "GluGluToSUEP_mS125.000_mPhi8.000_T32.000_modehadronic_13TeV_2018",
]
signal_labels = [
    "mS125_mPhi8_T16_leptonic",
    "mS125_mPhi8_T16_hadronic",
    "mS125_mPhi8_T32_leptonic",
    "mS125_mPhi8_T32_hadronic",
]

for i_r, region in enumerate(regions):
    hists_mc = []
    hist_bkg_total = None

    for process in mc_processes:
        h_mc = plots[process][region]
        print(process, h_mc.values())
        hists_mc.append(h_mc)
        if hist_bkg_total is None:
            hist_bkg_total = h_mc.copy()
        else:
            hist_bkg_total += h_mc.copy()

    hists_signal = []
    for process in signal_processes:
        h_signal = plots[process][region]
        hists_signal.append(h_signal)

    fig, ax = plt.subplots(figsize=(12.5, 11))

    hep.histplot(
        hists_mc,
        yerr=[np.sqrt(h.variances()) for h in hists_mc],
        stack=True,
        label=mc_labels,
        histtype="fill",
        ec="black",
        lw=2,
        ax=ax,
    )

    x_hatch = np.vstack(
        (hist_bkg_total.axes[0].edges[:-1], hist_bkg_total.axes[0].edges[1:])
    ).reshape((-1,), order="F")
    y_hatch1 = np.vstack((hist_bkg_total.values(), hist_bkg_total.values())).reshape(
        (-1,), order="F"
    )
    y_hatch1_unc = np.vstack(
        (np.sqrt(hist_bkg_total.variances()), np.sqrt(hist_bkg_total.variances()))
    ).reshape((-1,), order="F")
    ax.fill_between(
        x=x_hatch,
        y1=y_hatch1 - y_hatch1_unc,
        y2=y_hatch1 + y_hatch1_unc,
        label="Stat. Unc.",
        step="pre",
        facecolor="none",
        edgecolor=(0, 0, 0, 0.5),
        linewidth=0,
        hatch="///",
        zorder=2,
    )

    hep.histplot(
        hists_signal,
        yerr=[np.sqrt(h.variances()) for h in hists_signal],
        label=signal_labels,
        lw=3,
        ls="--",
        ax=ax,
    )

    h_data = plots[data][region]
    hep.histplot(
        h_data,
        label=["Data"],
        histtype="errorbar",
        mec="black",
        mfc="black",
        ecolor="black",
        ax=ax,
    )

    hep.cms.label(llabel="Preliminary", data=True, lumi=0.56, ax=ax)
    # hep.cms.label(llabel="Preliminary", data=True, lumi=55, ax=ax)
    plt.title(region)
    plt.gca().xaxis.set_minor_locator(plt.NullLocator())
    plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    plt.ylim(0.01 * y_limits[i_r][0], 0.01 * y_limits[i_r][1])
    plt.yscale("log")
    plt.legend(ncol=2)
    plt.xlabel("nMuon")
    plt.ylabel("events")
    plt.show()

In [ ]:
def export_histograms_to_root(plots, output_filename):
    """
    Export hist.Hist histograms to a ROOT file, organized in TDirectories by region.
    Negative bin entries are set to zero.

    Parameters:
    -----------
    plots : dict
        Nested dictionary containing hist.Hist objects
    output_filename : str
        Name of the output ROOT file
    """
    with uproot.recreate(output_filename) as f:
        for sample_name, regions in plots.items():
            for region_name, histogram in regions.items():
                # Create a copy to avoid modifying the original
                hist_copy = histogram.copy()

                # Set negative values to zero
                mask = hist_copy.values() < 0
                if mask.any():
                    hist_copy.view().value[mask] = 0
                    # hist_copy.view().variance[mask] = 0  # Also set corresponding variances to zero

                # Create the full path including directory
                path = f"{region_name}/{sample_name}"
                f[path] = hist_copy

In [ ]:
export_histograms_to_root(plots, "test.root")